# Assignment: Barcelona

## Prep

### Imports, shared definitions, datasets

In [ ]:
import geopandas as gpd
import pandas as pd
import matplotlib.pyplot as plt
import contextily
import pointpats
import numpy as np

In [ ]:
# load Barcelona listings for latest available date
listings_url = "https://data.insideairbnb.com/spain/catalonia/barcelona/2025-09-14/data/listings.csv.gz"
listings_df = pd.read_csv(listings_url, compression='gzip')
listings_df

In [ ]:
listings_geometry = gpd.points_from_xy(listings_df['longitude'], listings_df['latitude'], crs="EPSG:4326")
listings_geometry

In [ ]:
listings_gdf = gpd.GeoDataFrame(listings_df, geometry=listings_geometry)
listings_gdf.head()

In [ ]:
listings_gdf.crs

In [ ]:
listings_gdf.explore(tiles="CartoDB Positron")

# Price, plan of attack (roughly)

1. Find all points which have a price defined (ignore NaN)
2. Get the convex cull of the points
3. Apply a hexbin over this area, at some approx radius
4. Map each price to its hexbin
5. Compute median price of each hexbin (using median here as there are some very large prices)
6. For hexbin, apply spatial autocorrelation, Moran's I?



In [ ]:
def price_only(gdf):
    price_gdf = gdf.copy(deep=True)
    price_gdf["price"] = (
        gdf["price"]
          .str.replace(r"[\$,]", "", regex=True)
          .astype(float)
    )
    price_gdf = price_gdf[["price", gdf.geometry.name]]
    price_gdf = price_gdf.dropna()
    return price_gdf
    
listings_price_gdf = price_only(listings_gdf)
listings_price_gdf.head()

In [ ]:
listings_price_gdf.crs

In [ ]:
listings_price_gdf.explore("price", tiles="CartoDB Positron", scheme='percentiles')

In [ ]:
import h3

def point_to_h3_fn(h3_res):
    """returns a new function which will convert a POINT geometry into an H3 id"""
    def f(row):
        return h3.latlng_to_cell(row.geometry.y, row.geometry.x, h3_res)
    return f

def add_grid_cells(gdf, h3_res):
    """adds a new `h3_id` column to a GDF which is assumed to have a POINT geometry"""
    expected_crs = "EPSG:4326"
    assert gdf.crs == expected_crs, f"needed a CRS of {expected_crs}, but this was {gdf.crs}"
    gdf["h3_id"] = gdf.apply(point_to_h3_fn(h3_res), axis=1)
    
    

In [ ]:
add_grid_cells(listings_price_gdf, h3_res=10)
listings_price_gdf.head()

In [ ]:
from shapely.geometry import Polygon

def h3_to_polygon(h3_id):
    """takes an H3 and returns a boundary as a Shapely Polygon"""
    boundary = h3.cell_to_boundary(h3_id)
    lng_lat = [(lng, lat) for lat, lng in boundary]
    return Polygon(lng_lat)

def visualise_grid_cells(gdf):
    """takes a GDF with an `h3_id` column, or an index, which may contain duplicates, 
    and returns a new GDF with all the unique H3 cells as Polygons"""
    if gdf.index.name == 'h3_id':
        h3_ids = gdf.index
    else:
        h3_ids = gdf['h3_id']
    unique_ids = h3_ids.unique()
    polygon_gdf = gpd.GeoDataFrame(
        {'h3_id': unique_ids},
        geometry=[h3_to_polygon(h) for h in unique_ids],
        crs='EPSG:4326'
    )
    return polygon_gdf
    

In [ ]:
m = visualise_grid_cells(listings_price_gdf).explore(tiles="CartoDB Positron")
listings_price_gdf.explore("price", m=m, scheme="percentiles")
m

In [ ]:
# show price distribution

In [ ]:
import seaborn as sns

In [ ]:
sns.displot(listings_price_gdf["price"])

In [ ]:
# the distribution is very skewed so we'll use median rather than mean as a summary of each cell

In [ ]:
median_price_group = listings_price_gdf.groupby(["h3_id"])["price"].median()

In [ ]:
median_price_group

In [ ]:
type(median_price_group)

In [ ]:
def summary_price(gdf):
    """takes a GDF with a 'price' and 'h3_id' column and returns a new GDF with the median price per 'h3_id',
    and a geometry column which is the boundary of the cell as a Polygon"""
    median_price_group = gdf.groupby(["h3_id"])["price"].median()
    summary_gdf = gpd.GeoDataFrame(
        median_price_group,
        geometry=[h3_to_polygon(h) for h in median_price_group.index],
        crs='EPSG:4326'
    )
    return summary_gdf


In [ ]:
listings_price_summary_gdf = summary_price(listings_price_gdf)
listings_price_summary_gdf.head()

In [ ]:
listings_price_summary_gdf.explore("price", tiles="CartoDB Positron", scheme='percentiles')

In [ ]:
m = listings_price_summary_gdf.explore("price", tiles="CartoDB Positron", cmap="Blues", scheme="percentiles")
listings_price_gdf.explore("price", m=m, cmap="Reds", scheme="percentiles")
m

In [ ]:
listings_price_gdf.loc[listings_price_gdf["price"] == 210]

In [ ]:
listings_price_gdf.loc[listings_price_gdf["h3_id"] == "8a3944601037fff"]

In [ ]:
listings_price_summary_gdf.loc[listings_price_summary_gdf.index == "8a3944601037fff"]

In [ ]:
m = listings_price_summary_gdf.loc[listings_price_summary_gdf.index == "8a3944601037fff"].explore("price", tiles="CartoDB Positron", scheme="percentiles")
listings_price_gdf.loc[listings_price_gdf["h3_id"] == "8a3944601037fff"].explore("price", m=m, scheme="percentiles")
m